# Tutorial: Operadores Convolucionais em Processamento de Imagens e CNNs

> **Baseado em:** *Convolution: Image Filters, CNNs and Examples in Python & Pytorch* — Edward Roe  
> **Referências de código:** `raw_convolution.py`, `scipy_convolution.py`, `toy_autoencoder.py`

---

## Sumário

1. [Introdução](#1-Introdução)
2. [O que é Convolução?](#2-O-que-é-Convolução?)
3. [Filtros/Kernels Comuns](#3-Filtros/Kernels-Comuns)
4. [Implementação Manual da Convolução](#4-Implementação-Manual-da-Convolução)
5. [Convolução com SciPy](#5-Convolução-com-SciPy)
6. [Padding e Stride](#6-Padding-e-Stride)
7. [Convolução em CNNs](#7-Convolução-em-CNNs)
8. [Convolução 1×1](#8-Convolução-1×1)
9. [Convolução 3D](#9-Convolução-3D)
10. [Visualizando Filtros Aprendidos](#10-Visualizando-Filtros-Aprendidos)
11. [Convolução Transposta (Deconvolução)](#11-Convolução-Transposta)
12. [Autoencoder Convolucional com PyTorch](#12-Autoencoder-Convolucional-com-PyTorch)
13. [Convolução vs Correlação](#13-Convolução-vs-Correlação)
14. [Resumo e Considerações Finais](#14-Resumo-e-Considerações-Finais)

---
## 1. Introdução

A convolução bidimensional (2D) é amplamente conhecida no processamento digital de imagens para aplicar diversos filtros, como desfoque (*blur*), realce de nitidez e detecção de bordas. Com o advento das Redes Neurais Convolucionais (CNNs), as convoluções ganharam um novo interesse.

As convoluções são baseadas na ideia de usar um **filtro** (também chamado de *kernel*) e iterar sobre uma imagem de entrada para produzir uma imagem de saída.

**Neste tutorial você aprenderá:**
- A definição matemática de convolução 2D
- Como implementar convolução do zero em Python
- Filtros clássicos de processamento de imagem (Sobel, Laplacian, Emboss, etc.)
- Conceitos de *padding* e *stride*
- Como a convolução funciona dentro das CNNs
- Convolução transposta e sua aplicação em autoencoders

### Imports e Configurações Gerais

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal

# Para a parte de PyTorch
import torch
import torch.nn as nn

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

# Caminho da imagem de exemplo (ajuste para seu ambiente)
IMAGE_PATH = 'bj1_sml.jpg'

print('Bibliotecas carregadas com sucesso!')

---
## 2. O que é Convolução?

A convolução é um **operador linear** amplamente utilizado em processamento de sinais que, a partir de duas funções dadas, resulta em uma terceira que mede a soma do produto dessas funções ao longo do domínio implícito pela sua superposição.

No processamento digital de imagens, a convolução é um método matemático para combinar duas imagens produzindo uma terceira. Tipicamente, uma das duas imagens não é uma imagem em si, mas uma matriz cujo tamanho e valores determinam o efeito do processo de convolução — essa matriz é chamada de **filtro** ou **kernel**.

A ideia básica é: **alinhar o kernel sobre cada pixel da imagem e multiplicar e somar seus valores sobre o pixel e seus vizinhos locais**.

### Fórmula Matemática

A fórmula para convolução em duas dimensões é:

$$I'(u, v) = \sum_{i=-\infty}^{\infty} \sum_{j=-\infty}^{\infty} I(u-i, v-j) \cdot H(i,j) \tag{1}$$

onde $I$ e $H$ são funções bidimensionais, $(u, v)$ são as coordenadas do pixel na imagem e $(i, j)$ são as coordenadas do elemento do kernel.

Ou de forma mais prática, com o filtro $F$ de tamanho $(2N+1) \times (2N+1)$:

$$F * I(x, y) = \sum_{j=-N}^{N} \sum_{i=-N}^{N} F(i,j) \cdot I(x-i, y-j) \tag{2}$$

onde $(x, y)$ são as coordenadas do pixel na imagem e $(i, j)$ as coordenadas do elemento do kernel.

### Visualização do Processo de Convolução

> 📌 **[INSERIR IMAGEM]** — *Figura 1: Animação ilustrando como a convolução funciona em uma imagem 7×7. Sem padding e stride=1. A imagem resultante (direita) é menor que a original (esquerda), neste caso 5×5 em vez de 7×7.*

> 📌 **[INSERIR IMAGEM]** — *Figura 2: Exemplo do processo de convolução em processamento de imagem. Os resultados no centro são o desfoque de Sobel e a detecção de bordas. Normalmente, as imagens são convertidas para escala de cinza antes de aplicar um filtro.*

---
## 3. Filtros/Kernels Comuns

Dependendo do uso pretendido, os valores e tipos (inteiro, float) dos kernels podem variar. Abaixo estão os filtros mais comuns utilizados em processamento digital de imagens:

| Filtro | Finalidade |
|--------|------------|
| Blur / Suavização | Redução de ruído, borramento |
| Sobel X / Y | Detecção de bordas horizontais e verticais |
| Laplacian | Detecção de bordas em todas as direções |
| Emboss | Efeito de relevo |
| Prewitt | Detecção de bordas (alternativa ao Sobel) |

> 📌 **[INSERIR IMAGEM]** — *Figura 5: Exemplos de kernels usados em imagem digital. Ao lado de cada kernel há uma representação visual similar à usada em visualizações de pesos em CNNs.*

In [ ]:
# ============================================================
# Definição dos kernels clássicos
# ============================================================

# --- Blur (Suavização) ---
blur_3x3 = np.array([
    [1., 1., 1.],
    [1., 1., 1.],
    [1., 1., 1.]
], dtype="float") / 9.0   # normalizado: cada valor = 1/9

blur_5x5 = np.ones((5, 5), dtype="float") / 25.0

# --- Detecção de bordas: Sobel ---
Sobel_x = np.array([
    [ 1,  0, -1],
    [ 2,  0, -2],
    [ 1,  0, -1]
], dtype="float")

Sobel_y = np.array([
    [ 1,  2,  1],
    [ 0,  0,  0],
    [-1, -2, -1]
], dtype="float")

# --- Laplacian ---
laplacian = np.array([
    [1,  1, 1],
    [1, -8, 1],
    [1,  1, 1]
], dtype="float")

# --- Emboss ---
emboss = np.array([
    [-1, -1,  0],
    [-1,  0,  1],
    [ 0,  1,  1]
], dtype="float")

emboss2 = np.array([
    [-2, -1,  0],
    [-1,  1,  1],
    [ 0,  1,  2]
], dtype="float")

# --- Prewitt ---
prewitt = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype="float")

# --- Laplacian alternativo (scipy_convolution.py) ---
laplacian_4 = np.array([
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0]
], dtype="int")

kernels = {
    'blur_3x3': blur_3x3,
    'Sobel X': Sobel_x,
    'Sobel Y': Sobel_y,
    'Laplacian': laplacian,
    'Emboss': emboss,
    'Emboss 2': emboss2,
    'Prewitt': prewitt,
}

# Visualizando os kernels
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (name, kernel) in enumerate(kernels.items()):
    im = axes[idx].imshow(kernel, cmap='gray', vmin=kernel.min(), vmax=kernel.max())
    axes[idx].set_title(name, fontsize=13)
    axes[idx].axis('off')
    # Anotar valores
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            axes[idx].text(j, i, f'{kernel[i,j]:.2g}',
                           ha='center', va='center', fontsize=10,
                           color='red' if kernel[i,j] < 0 else 'white')

axes[-1].axis('off')  # oculta o último subplot vazio
plt.suptitle('Kernels Clássicos de Processamento de Imagem', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Implementação Manual da Convolução

A implementação a seguir mostra como a convolução funciona do zero — sem usar nenhuma biblioteca de alto nível. Não é a forma mais eficiente, mas é excelente para entender o processo.

**Ideia geral:**
1. Para cada pixel $(x, y)$ da imagem
2. Posicionar o kernel centrado nesse pixel
3. Multiplicar cada elemento do kernel pelo pixel correspondente da imagem
4. Somar todos os produtos → esse é o valor do pixel na imagem resultante

> 📌 **[INSERIR IMAGEM]** — *Figura 3: Exemplo de aplicação de filtro blur com convolução. Imagem original (esquerda), após convolução com blur_3×3 (centro) e blur_5×5 (direita). As bordas pretas mostram que a imagem resultante é menor que a original.*

In [ ]:
# ============================================================
# Convolução manual — baseada em raw_convolution.py
# ============================================================

def apply_filter(image_gray, kernel_x, kernel_y=None):
    """
    Aplica um filtro (ou par de filtros Sobel) à imagem em escala de cinza.

    Parâmetros:
        image_gray : np.ndarray — imagem em escala de cinza (uint8 ou float)
        kernel_x   : np.ndarray — kernel principal
        kernel_y   : np.ndarray ou None — kernel secundário (opcional, ex: Sobel Y)

    Retorna:
        filtered_image : np.ndarray — imagem filtrada
    """
    rows, columns = image_gray.shape
    filtered_image = np.zeros(shape=(rows, columns))
    kernel_dim = kernel_x.shape[0]

    for i in range(rows - (kernel_dim - 1)):
        for j in range(columns - (kernel_dim - 1)):
            patch = image_gray[i:i + kernel_dim, j:j + kernel_dim]
            gx = abs(np.sum(np.multiply(kernel_x, patch)))
            gy = 0
            if kernel_y is not None:
                gy = np.sum(np.multiply(kernel_y, patch))
            filtered_image[i + 1, j + 1] = np.sqrt(gx ** 2 + gy ** 2)

    return filtered_image


def emboss_filter(image_gray, kernel, factor=9, bias=128):
    """
    Aplica filtro de emboss com fator de escala e bias.

    Parâmetros:
        image_gray : np.ndarray — imagem em escala de cinza
        kernel     : np.ndarray — kernel do filtro
        factor     : int — divisor para normalizar o resultado
        bias       : int — valor adicionado para centralizar em 128 (cinza médio)

    Retorna:
        filtered_image : np.ndarray — imagem filtrada (uint8)
    """
    height, width = image_gray.shape
    filtered_image = np.zeros([height, width], dtype=np.uint8)
    height_k, width_k = kernel.shape
    init_x = width_k // 2
    init_y = height_k // 2

    for x in range(init_x, width - init_x):
        for y in range(init_y, height - init_y):
            resulting_pixel = 0
            for xk in range(width_k):
                for yk in range(height_k):
                    resulting_pixel += kernel[yk, xk] * image_gray[y + (yk - init_y), x + (xk - init_x)]
            filtered_image[y, x] = np.clip(int((resulting_pixel / factor) + bias), 0, 255)

    return filtered_image


print('Funções de convolução manual definidas!')

In [ ]:
# ============================================================
# Aplicando os filtros na imagem de exemplo
# ============================================================

# Carregando a imagem em escala de cinza
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Imagem não encontrada em: {IMAGE_PATH}")

img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# --- Aplicando cada filtro ---
results = {
    'Original (Cinza)': img_gray,
    'Blur 3×3':  apply_filter(img_gray, blur_3x3 * 9),   # kernel sem normalizar
    'Laplacian':  apply_filter(img_gray, laplacian),
    'Sobel X':    apply_filter(img_gray, Sobel_x),
    'Sobel Y':    apply_filter(img_gray, Sobel_y),
    'Sobel XY':   apply_filter(img_gray, Sobel_x, Sobel_y),
    'Emboss':     emboss_filter(img_gray, emboss),
    'Emboss 2':   emboss_filter(img_gray, emboss2),
}

# Visualização
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, (name, img) in enumerate(results.items()):
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(name, fontsize=13)
    axes[idx].axis('off')

plt.suptitle('Resultados da Convolução Manual', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

> 📌 **[INSERIR IMAGEM]** — *Figura 6: Exemplos de resultados da aplicação dos filtros usando os kernels da Figura 5. Linha superior: Emboss, Emboss 2, Laplacian. Linha inferior: Blur, Sobel X, Sobel Y.*

### Observações importantes sobre a implementação manual:

- A implementação usa **4 loops aninhados** — é correta, mas lenta para imagens grandes
- A imagem resultante é **menor** que a original quando não se usa padding (as bordas são ignoradas)
- Para o filtro Sobel combinado, calcula-se $\sqrt{G_x^2 + G_y^2}$
- Para filtros do tipo emboss, aplica-se um `bias` de 128 para que o cinza médio represente "sem mudança"

---
## 5. Convolução com SciPy

A biblioteca `scipy.signal` oferece `convolve()` — uma implementação eficiente da convolução. Compara-se aqui o uso do SciPy com os filtros já definidos.

In [ ]:
# ============================================================
# Convolução com SciPy — baseada em scipy_convolution.py
# ============================================================

# Kernels conforme scipy_convolution.py
kernels_scipy = {
    'Laplacian (scipy)': np.array([
        [ 0,  1,  0],
        [ 1, -4,  1],
        [ 0,  1,  0]
    ], dtype="int"),
    'Sobel X (scipy)': np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype="int"),
    'Sobel Y (scipy)': np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype="int"),
    'Emboss (scipy)': np.array([
        [ 0, -1,  0],
        [ 0,  0,  0],
        [ 0,  1,  0]
    ], dtype="int"),
    'Prewitt (scipy)': np.array([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1]
    ], dtype="int"),
}

# Carregando a imagem normalizada [0, 1]
img_norm = img_gray / 255.0

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

axes[0].imshow(img_norm, cmap='gray')
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

for idx, (name, kernel) in enumerate(kernels_scipy.items(), start=1):
    # mode='same' mantém a mesma dimensão da imagem de entrada
    filtered = signal.convolve(img_norm, kernel, mode='same')
    axes[idx].imshow(filtered, cmap='gray')
    axes[idx].set_title(name, fontsize=13)
    axes[idx].axis('off')

plt.suptitle('Convolução com scipy.signal.convolve (mode="same")', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Vantagem do SciPy: usa mode="same" → a saída tem as mesmas dimensões da entrada!')

### Diferença entre implementação manual e SciPy

| Aspecto | Manual (loops) | SciPy |
|---------|---------------|-------|
| Velocidade | Muito lenta | Rápida (FFT ou direto otimizado) |
| Tamanho da saída | Menor (sem padding) | Configurável via `mode` |
| Facilidade | Alta (didática) | Alta (produção) |
| `mode='same'` | Não disponível | Mantém dimensão de entrada |

---
## 6. Padding e Stride

**Padding** e **stride** são dois conceitos fundamentais em redes neurais convolucionais (CNNs) que afetam o tamanho dos mapas de features resultantes.

### 6.1 Padding

Como pode ser visto nos exemplos anteriores, cada imagem resultante é menor que a original por uma quantidade relacionada ao tamanho do kernel. Quanto maior o kernel, mais distante o centro fica da borda da imagem.

Para produzir uma saída com o **mesmo tamanho** da entrada, adicionamos pixels extras nas bordas. Dessa forma, ao deslizar, o kernel pode tratar os pixels originais das bordas como centro, enquanto os pixels extras ficam além da borda.

#### Tipos de Padding no OpenCV:

| Tipo | Descrição |
|------|----------|
| `BORDER_CONSTANT` | Preenche com valor constante (ex: zeros) |
| `BORDER_REFLECT` | Reflete os pixels da borda |
| `BORDER_REPLICATE` | Replica o pixel da borda |
| `BORDER_WRAP` | Envolve os pixels do lado oposto |

> 📌 **[INSERIR IMAGEM]** — *Figura 8: Quatro exemplos de padding usando OpenCV. (superior esquerdo) BORDER_CONSTANT em amarelo, (superior direito) BORDER_REFLECT, (inferior esquerdo) BORDER_REPLICATE, (inferior direito) BORDER_WRAP.*

> 📌 **[INSERIR IMAGEM]** — *Figura 9: Mesmo conceito da Figura 1, mas desta vez com padding de tamanho 1 adicionado (borda com zeros). A imagem resultante (direita) tem as mesmas dimensões que a original (5×5).*

In [ ]:
# ============================================================
# Demonstrando os tipos de Padding com OpenCV
# ============================================================

# Recorte pequeno para melhor visualização
img_crop = img_gray[:80, :80]
pad_size = 15

padding_types = {
    'BORDER_CONSTANT (zeros)': cv2.copyMakeBorder(img_crop, pad_size, pad_size, pad_size, pad_size,
                                                   cv2.BORDER_CONSTANT, value=0),
    'BORDER_REFLECT':          cv2.copyMakeBorder(img_crop, pad_size, pad_size, pad_size, pad_size,
                                                   cv2.BORDER_REFLECT),
    'BORDER_REPLICATE':        cv2.copyMakeBorder(img_crop, pad_size, pad_size, pad_size, pad_size,
                                                   cv2.BORDER_REPLICATE),
    'BORDER_WRAP':             cv2.copyMakeBorder(img_crop, pad_size, pad_size, pad_size, pad_size,
                                                   cv2.BORDER_WRAP),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (name, padded) in zip(axes, padding_types.items()):
    ax.imshow(padded, cmap='gray')
    ax.set_title(name, fontsize=11)
    # Destaca a borda de padding em vermelho
    rect = plt.Rectangle((pad_size - 0.5, pad_size - 0.5),
                          img_crop.shape[1], img_crop.shape[0],
                          edgecolor='red', facecolor='none', linewidth=2)
    ax.add_patch(rect)
    ax.axis('off')

plt.suptitle(f'Tipos de Padding (pad={pad_size}px) — região original em vermelho',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Stride

**Stride** é o número de pixels que o kernel desloca a cada passo sobre a imagem de entrada.

- **Stride = 1:** o kernel passa por todos os pixels sem pular nenhum
- **Stride = 2:** o kernel pula de 2 em 2 pixels → imagem resultante é menor

Quanto maior o stride, menor a imagem resultante.

> 📌 **[INSERIR IMAGEM]** — *Figura 10: Mesmo conceito da Figura 1, mas com padding=1 e stride=2. A imagem resultante (direita) tem dimensões 3×3, enquanto a original é 7×7.*

### Fórmula Geral do Tamanho de Saída

$$\text{output\_size} = \left\lfloor \frac{\text{input\_size} - \text{kernel\_size} + 2 \times \text{padding}}{\text{stride}} \right\rfloor + 1$$

In [ ]:
# ============================================================
# Calculadora de tamanho de saída
# ============================================================

def output_size(input_size, kernel_size, padding=0, stride=1):
    """Calcula o tamanho da saída de uma camada convolucional."""
    return int((input_size - kernel_size + 2 * padding) / stride) + 1

# Exemplos práticos
configs = [
    dict(input_size=224, kernel_size=3, padding=1, stride=1),
    dict(input_size=224, kernel_size=3, padding=0, stride=1),
    dict(input_size=224, kernel_size=3, padding=1, stride=2),
    dict(input_size=28,  kernel_size=5, padding=0, stride=1),
    dict(input_size=7,   kernel_size=3, padding=1, stride=2),
]

print(f"{'Input':>8} {'Kernel':>8} {'Padding':>9} {'Stride':>8} {'Output':>8}")
print('-' * 50)
for cfg in configs:
    out = output_size(**cfg)
    print(f"{cfg['input_size']:>8} {cfg['kernel_size']:>8} {cfg['padding']:>9} "
          f"{cfg['stride']:>8} {out:>8}")

---
## 7. Convolução em CNNs

Nas Redes Neurais Convolucionais (CNNs), a etapa de convolução é uma operação fundamental que aplica um conjunto de filtros (kernels) a uma imagem de entrada ou mapa de features para extrair características importantes, resultando em um **mapa de features** (*feature map*).

**O número de mapas de features resultantes corresponde ao número de filtros.**

> 📌 **[INSERIR IMAGEM]** — *Figura 11: Elementos participantes de uma camada convolucional. Imagem de entrada (esquerda), camada Conv2D com três kernels/filtros 3×3 (centro) e os três mapas de features resultantes (direita).*

### Diferença entre processamento de imagem tradicional e CNNs

| Aspecto | Processamento Tradicional | CNNs |
|---------|--------------------------|------|
| Kernels | Definidos manualmente | **Aprendidos** durante o treino |
| Inicialização | Valores fixos e conhecidos | Valores aleatórios |
| Canais | Geralmente escala de cinza | RGB separados |
| Objetivo | Filtros específicos | Extração automática de features |

Em CNNs, a convolução é realizada **separadamente para cada canal RGB** — diferente do processamento tradicional onde a imagem é convertida para escala de cinza.

In [ ]:
# ============================================================
# Simulando uma camada Conv2D com PyTorch
# ============================================================

import torch
import torch.nn as nn

# Convertendo a imagem para tensor PyTorch
# Formato esperado: [batch, canais, altura, largura]
img_tensor = torch.tensor(img_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
print(f"Shape do tensor de entrada: {img_tensor.shape}")
print(f"  → [batch={img_tensor.shape[0]}, canais={img_tensor.shape[1]}, "
      f"altura={img_tensor.shape[2]}, largura={img_tensor.shape[3]}]")

# Definindo uma camada Conv2D com kernel Sobel pré-configurado
conv_sobel = nn.Conv2d(in_channels=1, out_channels=2,
                        kernel_size=3, padding=1, bias=False)

# Substituindo os pesos pelo Sobel X e Sobel Y
with torch.no_grad():
    conv_sobel.weight[0, 0] = torch.tensor(Sobel_x, dtype=torch.float32)
    conv_sobel.weight[1, 0] = torch.tensor(Sobel_y, dtype=torch.float32)

# Forward pass
with torch.no_grad():
    output = conv_sobel(img_tensor)

print(f"\nShape da saída: {output.shape}")
print(f"  → [batch={output.shape[0]}, filtros={output.shape[1]}, "
      f"altura={output.shape[2]}, largura={output.shape[3]}]")

# Visualizando os 2 mapas de features resultantes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Entrada (escala de cinza)', fontsize=13)
axes[0].axis('off')

for i in range(2):
    fm = output[0, i].numpy()
    axes[i + 1].imshow(fm, cmap='gray')
    axes[i + 1].set_title(f'Feature Map {i+1} ({["Sobel X", "Sobel Y"][i]})', fontsize=13)
    axes[i + 1].axis('off')

plt.suptitle('Simulação de Camada Conv2D com Filtros Sobel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Convolução 1×1

Uma das desvantagens das redes de convolução profundas é que o **número de mapas de features frequentemente aumenta com a profundidade** da rede. Isso pode levar a um aumento significativo no número de parâmetros e na complexidade computacional, especialmente quando filtros grandes (5×5, 7×7) são utilizados.

A solução é usar um **filtro 1×1** para reduzir a profundidade ou o número de mapas de features. Foi introduzido no artigo *Network in Network* (Lin et al., 2013) e utilizado extensivamente na arquitetura Inception do Google.

### Como funciona a convolução 1×1:

- É uma operação **1×1×C**, onde $C$ é o número de canais/mapas de features
- **Não** inclui vizinhos do mesmo canal
- Pode ser aplicada como uma convolução 2D da esquerda para a direita e de cima para baixo com stride=1
- **Sem necessidade de padding**, resultando em um mapa de features com a mesma largura e altura da entrada
- É essencialmente uma **combinação linear dos canais** — mas pode-se adicionar não-linearidade aplicando funções de ativação

> 📌 **[INSERIR IMAGEM]** — *Figura 12: Exemplo de convolução 1×1×7 aplicada a um mapa de ativação 3×3×7 (esquerda), resultando em um mapa de ativação 3×3×1 (direita).*

In [ ]:
# ============================================================
# Demonstrando a convolução 1×1 com PyTorch
# ============================================================

# Simulando um mapa de features 3D: batch=1, canais=7, altura=3, largura=3
feature_map_3d = torch.randn(1, 7, 3, 3)
print(f"Shape do mapa de features de entrada: {feature_map_3d.shape}")
print(f"  → [batch, C_in=7, altura=3, largura=3]")

# Convolução 1×1: reduz de 7 canais para 1
conv_1x1 = nn.Conv2d(in_channels=7, out_channels=1,
                      kernel_size=1, padding=0, bias=False)

with torch.no_grad():
    out_1x1 = conv_1x1(feature_map_3d)

print(f"\nShape da saída após Conv 1×1: {out_1x1.shape}")
print(f"  → [batch, C_out=1, altura=3, largura=3]")
print(f"\nDimensões espaciais PRESERVADAS: {feature_map_3d.shape[2:]} → {out_1x1.shape[2:]}")
print(f"Profundidade REDUZIDA: {feature_map_3d.shape[1]} → {out_1x1.shape[1]} canal")

# Contagem de parâmetros
n_params = sum(p.numel() for p in conv_1x1.parameters())
print(f"\nParâmetros do kernel 1×1: {n_params} (= 7 pesos × 1 filtro)")

# Comparando com 3×3
conv_3x3 = nn.Conv2d(in_channels=7, out_channels=1, kernel_size=3, padding=0, bias=False)
n_params_3x3 = sum(p.numel() for p in conv_3x3.parameters())
print(f"Parâmetros de um kernel 3×3: {n_params_3x3} (= 7 × 3 × 3 pesos × 1 filtro)")
print(f"Redução: {n_params_3x3 // n_params}× menos parâmetros com 1×1!")

---
## 9. Convolução 3D

Em todos os exemplos anteriores, a convolução foi aplicada a imagens ou matrizes bidimensionais. A mesma ideia funciona para **matrizes tridimensionais**.

No caso de uma matriz 3D (como frequentemente ocorre nas convoluções em modelos CNN), **o kernel também será uma matriz 3D**. O resultado é um único pixel a cada iteração.

> 📌 **[INSERIR IMAGEM]** — *Figura 13: Exemplo de convolução de uma matriz 3D. O kernel é um cubo 3×3×3 (esquerda), sem padding e stride=1. A animação mostra como o kernel percorre o array tridimensional (direita).*

Os conceitos de padding e stride se aplicam igualmente às convoluções tridimensionais.

In [ ]:
# ============================================================
# Demonstrando Conv3D com PyTorch
# ============================================================

# Entrada 3D: batch=1, canais=1, profundidade=5, altura=5, largura=5
input_3d = torch.randn(1, 1, 5, 5, 5)
print(f"Shape da entrada 3D: {input_3d.shape}")
print(f"  → [batch=1, C=1, prof=5, altura=5, largura=5]")

# Kernel 3×3×3
conv3d = nn.Conv3d(in_channels=1, out_channels=1,
                    kernel_size=3, padding=0, stride=1, bias=False)

with torch.no_grad():
    output_3d = conv3d(input_3d)

print(f"\nShape da saída 3D: {output_3d.shape}")
print(f"  → [batch=1, C=1, prof=3, altura=3, largura=3]")
print(f"\nCálculo: output = (5 - 3 + 0) / 1 + 1 = 3 em cada dimensão")

# Com padding
conv3d_pad = nn.Conv3d(in_channels=1, out_channels=1,
                        kernel_size=3, padding=1, stride=1, bias=False)
with torch.no_grad():
    output_3d_pad = conv3d_pad(input_3d)

print(f"\nCom padding=1: {output_3d_pad.shape}")
print(f"  → Dimensões preservadas com padding!")

---
## 10. Visualizando Filtros Aprendidos

As redes neurais são geralmente inicializadas com **valores aleatórios**. No caso das CNNs, esses valores iniciais são os elementos do filtro (kernels). À medida que o treinamento evolui, esses filtros assumem um aspecto mais "organizado", sendo possível reconhecer um certo padrão em cada um deles.

> 📌 **[INSERIR IMAGEM]** — *Figura 14: Alguns kernels das primeiras camadas convolucionais do modelo ResNet-50. Kernels não treinados parecem aleatórios, enquanto os treinados mostram padrões distintos.*

> 📌 **[INSERIR IMAGEM]** — *Figura 15: Alguns mapas de features resultantes das primeiras camadas convolucionais do modelo VGG-16. Note que alguns mapas resultantes dos kernels treinados (direita) se assemelham ao resultado da aplicação do filtro emboss.*

> 📌 **[INSERIR IMAGEM]** — *Figura 16: Exemplo de mapa de features resultante de cada uma das 13 camadas convolucionais do VGG-16. As imagens foram redimensionadas para melhor visualização.*

> 📌 **[INSERIR IMAGEM]** — *Figura 17: Evolução do tamanho dos mapas de features durante o forward pass no modelo VGG-16: 224×224 → 112×112 → 56×56 → 28×28 → 14×14.*

In [ ]:
# ============================================================
# Visualizando filtros aleatórios vs. filtros conhecidos
# (simulação do comportamento treinado vs. não treinado)
# ============================================================

torch.manual_seed(42)

# Kernels não treinados (aleatórios)
conv_untrained = nn.Conv2d(1, 9, kernel_size=3, padding=1, bias=False)

# Kernels 'treinados' = filtros clássicos bem conhecidos
known_kernels = [
    Sobel_x, Sobel_y, laplacian,
    emboss, emboss2, prewitt,
    blur_3x3 * 9, Sobel_x * -1, Sobel_y * -1
]

fig, axes = plt.subplots(2, 9, figsize=(20, 5))

for i in range(9):
    # Não treinado
    w_untrained = conv_untrained.weight[i, 0].detach().numpy()
    axes[0, i].imshow(w_untrained, cmap='gray')
    axes[0, i].axis('off')
    if i == 4:
        axes[0, i].set_title('← Não Treinado →', fontsize=11, color='blue')

    # 'Treinado' (filtros conhecidos)
    axes[1, i].imshow(known_kernels[i], cmap='gray')
    axes[1, i].axis('off')
    if i == 4:
        axes[1, i].set_title('← Filtros Clássicos →', fontsize=11, color='green')

plt.suptitle('Kernels Aleatórios (não treinados) vs. Filtros Clássicos (como kernels aprendidos)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Convolução Transposta

A **camada de convolução transposta** (também chamada incorretamente de *deconvolução*) tem como objetivo **aumentar as dimensões** (altura e largura) de suas entradas.

### Diferença entre Deconvolução e Convolução Transposta:
- **Deconvolução:** reverte a operação de convolução e recupera a entrada original (por valores)
- **Convolução Transposta:** reverte a convolução **não pelos valores, mas pelas dimensões**

A convolução transposta pode ser considerada como uma convolução padrão com um **mapa de features de entrada modificado**.

### Como funciona:
Para modificar o mapa de features de entrada, **sequências de zeros são inseridas nas colunas e linhas** entre os valores existentes. O número de colunas/linhas a inserir é indicado por `stride - 1`.

> 📌 **[INSERIR IMAGEM]** — *Figura 18: Exemplo de modificação do mapa de features de entrada para convolução transposta. Imagem original 2×2 → zeros adicionados (stride-1) → padding com zeros.*

#### Fórmula do tamanho de saída (ConvTranspose2d):
$$\text{output\_size} = (\text{input\_size} - 1) \times \text{stride} - 2 \times \text{padding} + \text{kernel\_size}$$

In [ ]:
# ============================================================
# Demonstrando Convolução Transposta (upsampling)
# ============================================================

def transposed_output_size(input_size, kernel_size, stride=1, padding=0):
    """Calcula o tamanho de saída de uma convolução transposta."""
    return (input_size - 1) * stride - 2 * padding + kernel_size

# Demonstração numérica
print("Convolução NORMAL (reduz dimensões):")
print(f"  Input 6×6, kernel 3×3, stride=3, padding=1")
print(f"  → Saída: {output_size(6, 3, padding=1, stride=3)}×{output_size(6, 3, padding=1, stride=3)}")

print("\nConvolução TRANSPOSTA (aumenta dimensões):")
print(f"  Input 2×2, kernel 3×3, stride=3, padding=1")
out = transposed_output_size(2, 3, stride=3, padding=1)
print(f"  → Saída: {out}×{out}")

# Usando PyTorch
print("\n--- Verificando com PyTorch ---")
x_small = torch.randn(1, 64, 6, 6)
print(f"Entrada: {x_small.shape}")

# Convoluções normais (encoder): 6×6 → 2×2
conv = nn.Conv2d(64, 32, kernel_size=3, stride=3, padding=1)
with torch.no_grad():
    x_encoded = conv(x_small)
print(f"Após Conv2d (stride=3): {x_encoded.shape}")

# Convolução transposta (decoder): 2×2 → 6×6
conv_t = nn.ConvTranspose2d(32, 64, kernel_size=3, stride=3, padding=1)
with torch.no_grad():
    x_decoded = conv_t(x_encoded)
print(f"Após ConvTranspose2d (stride=3): {x_decoded.shape}")

---
## 12. Autoencoder Convolucional com PyTorch

O código a seguir ilustra um **autoencoder convolucional** simples — treinado no dataset STL10 — apenas para demonstrar o uso de convoluções transpostas. Um autoencoder aprende a comprimir (encoder) e reconstruir (decoder) imagens.

> 📌 **[INSERIR IMAGEM]** — *Figura 19: Evolução de um mapa de features pelas seis camadas do modelo treinado: 91×91 → 31×31 → 11×11 → 6×6 (encoder) → 11×11 → 31×31 → 91×91 (decoder).*

> 📌 **[INSERIR IMAGEM]** — *Figura 20: Comparação entre os dois extremos da convolução transposta. A imagem 6×6 não se assemelha minimamente ao beija-flor original.*

In [ ]:
# ============================================================
# Autoencoder Convolucional — baseado em toy_autoencoder.py
# ============================================================

class ConvAutoEncoder(nn.Module):
    """
    Autoencoder convolucional simples para imagens em escala de cinza 91×91.

    Arquitetura:
      Encoder: 3 camadas Conv2d com ReLU
        91×91 → 31×31 → 11×11 → 6×6 (feature maps)

      Decoder: 3 camadas ConvTranspose2d com ReLU/Tanh
        6×6 → 11×11 → 31×31 → 91×91
    """

    def __init__(self):
        super().__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 3), stride=(3, 3), padding=(1, 1)),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=(3, 3), stride=(3, 3), padding=(1, 1)),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)),
            nn.ReLU(True),
        )

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(3, 3), padding=(1, 1)),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 1, kernel_size=(3, 3), stride=(3, 3), padding=(1, 1)),
            nn.Tanh()  # Saída normalizada entre -1 e 1
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


# Instanciar e inspecionar
model = ConvAutoEncoder()
print(model)

# Verificando as dimensões com um forward pass
dummy = torch.randn(1, 1, 91, 91)  # batch=1, canal=1, 91×91

print("\n--- Dimensões por camada (Encoder) ---")
x = dummy
print(f"Entrada:   {x.shape}")
for i, layer in enumerate(model.encoder):
    x = layer(x)
    if isinstance(layer, nn.Conv2d):
        print(f"Após Conv2d [{i}]: {x.shape}")

print(f"\n--- Dimensões por camada (Decoder) ---")
print(f"Entrada decoder (bottleneck): {x.shape}")
for i, layer in enumerate(model.decoder):
    x = layer(x)
    if isinstance(layer, (nn.ConvTranspose2d, nn.Tanh)):
        print(f"Após ConvTranspose2d/Tanh [{i}]: {x.shape}")

In [ ]:
# ============================================================
# Contagem de parâmetros do modelo
# ============================================================

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)
print(f"Total de parâmetros:     {total:,}")
print(f"Parâmetros treináveis:   {trainable:,}")

# Resumo por camada
print("\n--- Parâmetros por camada ---")
for name, param in model.named_parameters():
    print(f"  {name:45s}: {param.numel():>8,} params  {list(param.shape)}")

In [ ]:
# ============================================================
# Loop de treinamento simplificado (toy example)
# Nota: para uso real, utilize o script toy_autoencoder.py
# com o dataset STL10 completo
# ============================================================

from torchvision import transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")

model = ConvAutoEncoder().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

# Criando dado sintético para demonstração (sem dataset real)
img_transform = transforms.Compose([
    transforms.Normalize([0.5], [0.5])  # normaliza para [-1, 1]
])

# Imagem real como dado de exemplo
img_resized = cv2.resize(img_gray, (91, 91)) / 255.0
img_t = torch.tensor(img_resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # [1,1,91,91]
img_t = img_transform(img_t).to(device)

# Treino por 30 iterações
losses = []
model.train()
for step in range(30):
    optimizer.zero_grad()
    output = model(img_t)
    loss = loss_fn(output, img_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (step + 1) % 10 == 0:
        print(f"Step {step+1:3d}/30 — Loss: {loss.item():.6f}")

# Visualizando original vs. reconstruída
model.eval()
with torch.no_grad():
    reconstructed = model(img_t).squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_resized, cmap='gray')
axes[0].set_title('Original (91×91)', fontsize=13)
axes[0].axis('off')

# Bottleneck
with torch.no_grad():
    bottleneck = model.encoder(img_t)[0, 0].cpu().numpy()
axes[1].imshow(bottleneck, cmap='gray')
axes[1].set_title(f'Bottleneck (6×6, canal 0)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(reconstructed, cmap='gray')
axes[2].set_title('Reconstruída (91×91)', fontsize=13)
axes[2].axis('off')

plt.suptitle('Autoencoder: Original → Bottleneck → Reconstruída\n(apenas 30 steps de treino)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Curva de loss
plt.figure(figsize=(8, 4))
plt.plot(losses, 'b-o', markersize=4)
plt.title('Curva de Loss (MSE) — Treinamento')
plt.xlabel('Iteração')
plt.ylabel('Loss (MSE)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 13. Convolução vs Correlação

Convolução e correlação são operações matemáticas similares em conceito, mas com uma **diferença fundamental**: o tratamento do kernel.

### Convolução
- O filtro/kernel é **rotacionado 180°** antes da operação
- O kernel invertido é então deslizado sobre o sinal de entrada
- Em cada posição, realiza-se a multiplicação elemento a elemento entre o filtro e a porção correspondente da entrada

### Correlação (Cross-Correlation)
- O filtro/kernel **não é invertido**
- É deslizado diretamente sobre o sinal de entrada
- Em cada posição, realiza-se a multiplicação elemento a elemento

Em ambos os casos, os produtos resultantes são somados para obter um único valor.

> **Nota prática:** Se o filtro for **simétrico**, os resultados de correlação e convolução são idênticos. Na maioria das implementações de deep learning (incluindo PyTorch), a operação chamada de "convolução" é tecnicamente uma **correlação cruzada** — mas convencionou-se chamar de convolução.

In [ ]:
# ============================================================
# Convolução vs Correlação — demonstração numérica
# ============================================================

# Imagem simples 5×5
img_simple = np.array([
    [1, 2, 3, 0, 0],
    [4, 5, 6, 0, 0],
    [7, 8, 9, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
], dtype=float)

# Kernel assimétrico para evidenciar a diferença
kernel_asym = np.array([
    [1, 0, -1],
    [2, 0, -2],
    [0, 0,  0],
], dtype=float)

# Kernel rotacionado 180°
kernel_rotated = np.rot90(kernel_asym, 2)

# Correlação: sem rotação do kernel (padrão do PyTorch/TF)
correlation = signal.correlate2d(img_simple, kernel_asym, mode='same')

# Convolução: com rotação do kernel
convolution_result = signal.convolve2d(img_simple, kernel_asym, mode='same')

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(img_simple, cmap='Blues')
axes[0].set_title('Imagem Original', fontsize=13)
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, int(img_simple[i, j]), ha='center', va='center', fontsize=11)
axes[0].axis('off')

axes[1].imshow(kernel_asym, cmap='RdBu')
axes[1].set_title('Kernel (assimétrico)', fontsize=13)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, int(kernel_asym[i, j]), ha='center', va='center', fontsize=11)
axes[1].axis('off')

axes[2].imshow(correlation, cmap='gray')
axes[2].set_title('Correlação\n(sem rotação do kernel)', fontsize=13)
for i in range(5):
    for j in range(5):
        axes[2].text(j, i, f'{correlation[i,j]:.0f}', ha='center', va='center', fontsize=9)
axes[2].axis('off')

axes[3].imshow(convolution_result, cmap='gray')
axes[3].set_title('Convolução\n(com rotação 180° do kernel)', fontsize=13)
for i in range(5):
    for j in range(5):
        axes[3].text(j, i, f'{convolution_result[i,j]:.0f}', ha='center', va='center', fontsize=9)
axes[3].axis('off')

plt.suptitle('Convolução vs Correlação com Kernel Assimétrico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("São iguais?", np.allclose(correlation, convolution_result))

# Com kernel simétrico
kernel_sym = np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]], dtype=float)  # Laplacian (simétrico)
corr_sym = signal.correlate2d(img_simple, kernel_sym, mode='same')
conv_sym = signal.convolve2d(img_simple, kernel_sym, mode='same')
print(f"Com kernel SIMÉTRICO (Laplacian), são iguais? {np.allclose(corr_sym, conv_sym)}")

---
## 14. Resumo e Considerações Finais

Este tutorial abordou os seguintes tópicos relacionados à convolução:

| Tópico | Conceito-chave |
|--------|---------------|
| **Conceito de Convolução** | Operação linear que desliza um kernel sobre a imagem |
| **Imagens Cinza vs. Coloridas** | Cinza = 1 canal; RGB = 3 canais processados separadamente |
| **Kernel, Stride, Padding** | Definem o comportamento e o tamanho da saída |
| **Convolução 2D** | Base para filtros clássicos e CNNs |
| **Filtros Clássicos** | Blur, Sobel, Laplacian, Emboss, Prewitt |
| **Convolução 1×1** | Reduz profundidade sem alterar dimensões espaciais |
| **Convolução 3D** | Mesma ideia para matrizes tridimensionais |
| **Convolução Transposta** | Aumenta dimensões — usada em decoders/geradores |
| **Autoencoder Conv.** | Demonstra encoder (compressão) + decoder (expansão) |
| **Conv. vs Correlação** | Diferença: rotação do kernel (kernels simétricos → resultado idêntico) |

### Fórmulas importantes

$$\text{Convolução 2D: } F * I(x, y) = \sum_{j=-N}^{N} \sum_{i=-N}^{N} F(i,j) \cdot I(x-i, y-j)$$

$$\text{Tamanho de saída (Conv): } \left\lfloor \frac{\text{input} - \text{kernel} + 2 \cdot \text{pad}}{\text{stride}} \right\rfloor + 1$$

$$\text{Tamanho de saída (ConvTranspose): } (\text{input} - 1) \cdot \text{stride} - 2 \cdot \text{pad} + \text{kernel}$$

---
## Referências

[1] Gonzalez, R. C., Woods, R. E., *Digital Image Processing*. 4ª edição. Pearson Prentice Hall, 2017.

[2] Burger, W. e Burge, M. J., *Digital Image Processing: An Algorithmic Introduction using Java*. Springer, 1ª edição, 2008.

[3] Lin, M., Chen, Q. e Yan, S., Sermanet, P., Szegedy, C., *Network in network*. CoRR, abs/1312.4400, 2013.

[4] Szegedy, C., Liu, W., Jia, Y., Reed, S., Anguelov, D., Erhan, D., Vanhoucke, V., Rabinovich, A., *Going Deeper with Convolutions*, arXiv:1409.4842v1 [cs.CV], 2014.

[5] Roe, E., *Convolution: Image Filters, CNNs and Examples in Python & Pytorch*. Medium, 2023.

---
*Tutorial elaborado com base nos materiais do curso, scripts Python fornecidos (`raw_convolution.py`, `scipy_convolution.py`, `toy_autoencoder.py`) e no artigo de Edward Roe.*